In [1]:
!pip install transformers torch

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
chat_history_ids = None

def generate_response(user_input, chat_history_ids):
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    if chat_history_ids is not None:
        input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        input_ids = new_input_ids

    chat_history_ids = model.generate(
        input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.75
    )

    response = tokenizer.decode(chat_history_ids[:, input_ids.shape[-1]:][0], skip_special_tokens=True)

    return response, chat_history_ids

In [6]:
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day!")
        break

    response, chat_history_ids = generate_response(user_input, chat_history_ids)

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. How can I help you today?
You: Hello


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: Hi! : 3
You: Hi.
Chatbot: I love you
You: What is Artificial Intelligence?
Chatbot: It's a meme you dip
You: Who created Python?
Chatbot: Who is that who is the person who is the person who is the person who is the person who is the person who is the person who is the person who is the person?


KeyboardInterrupt: Interrupted by user

In [7]:
model_name = "gpt2"

In [8]:
def generate_response(user_input, chat_history_ids):
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7,
        no_repeat_ngram_size=3
    )

    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    return response, chat_history_ids

In [9]:
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day!")
        break

    response, chat_history_ids = generate_response(user_input, chat_history_ids)

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. How can I help you today?


KeyboardInterrupt: Interrupted by user

In [10]:
from transformers import pipeline

chatbot = pipeline("text-generation", model="gpt2")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    response = chatbot(user_input, max_length=50, num_return_sequences=1)

    print("Chatbot:", response[0]['generated_text'])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chatbot: Who created Python?

Python is a programming language, and it's not a programming language that needs to be reinvented every time you want to learn how to program. It's a programming language that can be used as a teaching tool for anyone, whether you're trying to learn about programming or just trying to learn Python.

A good way to approach learning Python is to learn about learning Python by having someone from the Python community help you learn it. If you're learning Python, you need to know about Python by learning about learning Python by learning Python by learning Python.

If you want to learn Python, you have two ways to do it. First, you can use Python's documentation or learn about Python by learning about learning Python by learning about learning Python by learning Python. Second, you can learn about Python by using Python's source code and by learning about learning Python by learning about learning Python by learning Python by learning Python.

It's important t

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chatbot: Who is the creator of Python? Why is it the subject of a New York Times best-seller?

This interview was conducted on April 10, 2013.

The question I have is about how we can all learn to share ideas and how to take them to the next level and help others.

Some of my mentors in computing, for example, are people I know intimately, who are experts in computer science and computer science and I am a software engineer. And they all know how to share their ideas and how to make it better.

I think some of them may have been inspired by the success of the Raspberry Pi, which is a great and powerful piece of software that I've been working on since I was a teenager. I've been working on the project for four years now. I've been working with people from all over the world and have a lot of great people, and they all share ideas. It's a very diverse community. So I think it's important that we all share ideas.

But I've been doing this with people of all backgrounds who've been expose

KeyboardInterrupt: Interrupted by user

In [11]:
!pip install transformers torch

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

chat_history_ids = None

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
def generate_response(user_input, chat_history_ids):
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    chat_history_ids = model.generate(
        bot_input_ids,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7,
        no_repeat_ngram_size=3
    )

    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    return response, chat_history_ids

In [14]:
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day!")
        break

    response, chat_history_ids = generate_response(user_input, chat_history_ids)

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. How can I help you today?
You: Hello.
Chatbot: Hello.
You: What is Artificial Intelligence?
Chatbot: It's a thing.
You: Wow. Who created Python? A human?
Chatbot: No, it was a thing created by humans.


KeyboardInterrupt: Interrupted by user

In [17]:
def generate_response(user_input, chat_history_ids):

    prompt = "You are a helpful and intelligent AI assistant. Answer clearly and accurately.\nUser: " + user_input + "\nBot:"

    new_input_ids = tokenizer.encode(prompt + tokenizer.eos_token, return_tensors='pt')

    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    chat_history_ids = model.generate(
        bot_input_ids,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.6,
        no_repeat_ngram_size=3
    )

    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    return response, chat_history_ids

In [18]:
model_name = "microsoft/DialoGPT-large"

In [19]:
user_input = input("You: ")

You: Hello


In [20]:
!pip install gradio

I am using AI to get a good UI for the chatbot from here...

In [21]:
import gradio as gr

chat_history_ids = None

def chatbot_response(user_input):
    global chat_history_ids

    prompt = "You are a helpful AI assistant.\nUser: " + user_input + "\nBot:"

    new_input_ids = tokenizer.encode(prompt + tokenizer.eos_token, return_tensors='pt')

    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    chat_history_ids = model.generate(
        bot_input_ids,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.9,
        temperature=0.6
    )

    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    return response


interface = gr.Interface(
    fn=chatbot_response,
    inputs="text",
    outputs="text",
    title="AI Chatbot"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://983492cbe4c0a4c0ad.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
